<img src="http://imgur.com/1ZcRyrc.png" style="float: left; margin: 20px; height: 55px">

# Lesson 6.02 - Bootstrapping and Bagging

## Importing libraries


We'll need the following libaries for today's lecture:
- `pandas`
- `numpy`
- `DecisionTreeClassifier` from `sklearn`'s `tree` module
- `BaggingClassifier` from `sklearn`'s `ensemble` module
- `train_test_split` and `cross_val_score` from `sklearn`'s `model_selection` module
- `accuracy_score` from `sklearn`'s `metrics` module

In [1]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score

/Users/rileydallas/anaconda3/lib/python3.6/site-packages/sklearn/ensemble/weight_boosting.py:29: DeprecationWarning: numpy.core.umath_tests is an internal NumPy module and should not be imported. It will be removed in a future NumPy release.
  from numpy.core.umath_tests import inner1d


## Load the Data

We'll be using the `Heart.csv` from the [ISLR Website](https://www-bcf.usc.edu/~gareth/ISL/). There's a copy in this repo under `./datasets/Heart.csv`.

**Drop the `Unnamed: 0` column once you've loaded the dataset.

In [2]:
df = pd.read_csv('datasets/Heart.csv')
df.drop('Unnamed: 0', axis=1, inplace=True)
df.head()

,Age,Sex,ChestPain,RestBP,Chol,Fbs,RestECG,MaxHR,ExAng,Oldpeak,Slope,Ca,Thal,AHD
0,63,1,typical,145,233,1,2,150,0,2.3,3,0.0,fixed,No
1,67,1,asymptomatic,160,286,0,2,108,1,1.5,2,3.0,normal,Yes
2,67,1,asymptomatic,120,229,0,2,129,1,2.6,2,2.0,reversable,Yes
3,37,1,nonanginal,130,250,0,0,187,0,3.5,3,0.0,normal,No
4,41,0,nontypical,130,204,0,2,172,0,1.4,1,0.0,normal,No


## Data cleaning: Drop rows with null values

In [3]:
df.shape

(303, 14)

In [4]:
df.dropna(inplace=True)

In [5]:
df.shape

(297, 14)

## Feature Engineering: Create dummies

Dummy the following columns:
- `ChestPain`
- `Thal`
- `AHD`

Be sure to set `drop_first=True`.

In [6]:
df = pd.get_dummies(df, columns=['ChestPain', 'Thal', 'AHD'], drop_first=True)
df.head()

,Age,Sex,RestBP,Chol,Fbs,RestECG,MaxHR,ExAng,Oldpeak,Slope,Ca,ChestPain_nonanginal,ChestPain_nontypical,ChestPain_typical,Thal_normal,Thal_reversable,AHD_Yes
0,63,1,145,233,1,2,150,0,2.3,3,0.0,0,0,1,0,0,0
1,67,1,160,286,0,2,108,1,1.5,2,3.0,0,0,0,1,0,1
2,67,1,120,229,0,2,129,1,2.6,2,2.0,0,0,0,0,1,1
3,37,1,130,250,0,0,187,0,3.5,3,0.0,1,0,0,1,0,0
4,41,0,130,204,0,2,172,0,1.4,1,0.0,0,1,0,1,0,0


## Model Prep: Create `X` and `y` variables

Our target column will be `AHD_Yes`: 
- 1 means the patient has heart disease
- 0 means they aren't diagnosed with heart disease

In [7]:
X = df.drop('AHD_Yes', axis=1)
y = df['AHD_Yes']

## Challenge: What is the baseline accuracy?

In [8]:
y.value_counts(normalize=True)

0    0.538721
1    0.461279
Name: AHD_Yes, dtype: float64

## Model Prep: Train/Test Split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

## Model Instantiation

Create an instance of `DecisionTreeClassifier`.

In [10]:
tree = DecisionTreeClassifier()

## Model Evaluation

In the cells below:
1. Get a `cross_val_score` for our `tree`
2. Fit and score on the training data
3. Score on the test data

Where do Decision Trees fall on the Bias/Variance spectrum?

In [11]:
cross_val_score(tree, X_train, y_train).mean()

0.6801801801801801

In [12]:
tree.fit(X_train, y_train)
tree.score(X_train, y_train)

1.0

In [13]:
tree.score(X_test, y_test)

0.7333333333333333

## Introduction to Ensemble Methods
We can list out the different types of models we've built thus far:
- Linear Regression
- Generalized Linear Models
    - Logistic Regression
    - Multinomial Logistic Regression
    - Poisson Regression
- $k$-Nearest Neighbors
- Naive Bayes Classification

If we want to use any of these models, we follow the same type of process.
1. Based on our problem, we identify which model to use. (Is our problem classification or regression? Do we want an interpretable model?)
2. Fit the model using the training data.
3. Use the fit model to generate predictions.
4. Evaluate our model's performance and, if necessary, return to step 2 and make changes.

In this case, we've always had exactly one model. Today, however, we're going to talk about **ensemble methods**. Mentally, you should think about this as if we build multiple models and then aggregate their results in some way.

## Why would we build an "ensemble model?"
Consider $H$ the space of all possible hypotheses. Our goal is to estimate $f$, the true function. We can come up with different hypotheses $h_1$, $h_2$, and so on to get as close to $f$ as possible. 
- Think about $f$ as the true process that dictates Iowa liquor sales.
- Think about $h_1$ as the model you built to predict $f$.

![](./assets/images/ensemble-benefits.png)

- The **statistical** benefit to ensemble methods: By building one model, our predictions are almost certainly going to be wrong. Predictions from one model might overestimate liquor sales; predictions from another model might underestimate liquor sales. By "averaging" predictions from multiple models, we'll see that we can often cancel our errors out and get closer to the true function $f$.
- The **computational** benefit to ensemble methods: It might be impossible to develop one model that globally optimizes our objective function. (Remember that CART reach locally-optimal solutions and that all generalized linear models iterate toward a solution that isn't guaranteed to be the globally-optimal solution.) In these cases, it may be impossible for one CART or an individual GLM to arrive at the true function $f$. However, starting our "local searches" at different points and aggregating our predictions might .
- The **representational** benefit to ensemble methods: Even if we had all the data and all the computer power in the world, it might be impossible for one model to exactly equal $f$. For example, a linear regression model can never model a relationship where a one-unit change in $X$ effects some *different* change in $Y$ based on the value of $X$. All models have some shortcomings. (See [the no free lunch theorems](https://en.wikipedia.org/wiki/No_free_lunch_in_search_and_optimization).) While individual models have shortcomings, by creating multiple models and aggregating their predictions, we can actually create predictions that represent something that one model cannot ever represent.

## Exercise: Guess the weight of Penelope

![](./assets/penelope.jpg)

[Image source: https://www.npr.org](https://www.npr.org/sections/money/2015/07/17/422881071/how-much-does-this-cow-weigh)

## Ensemble predictions

Let's mimic the "wisdom of the crowds" by creating several Decision Trees and averaging their predictions on the test set.

In [14]:
predictions = pd.DataFrame(index=X_test.index)

for i in range(1,11):
    t = DecisionTreeClassifier()
    t.fit(X_train, y_train)
    predictions['Tree {}'.format(i)] = t.predict(X_test)
predictions

,Tree 1,Tree 2,Tree 3,Tree 4,Tree 5,Tree 6,Tree 7,Tree 8,Tree 9,Tree 10
113,1,1,1,1,1,1,1,1,1,1
195,1,1,1,1,1,1,1,1,1,1
64,1,1,1,1,1,1,1,1,1,1
27,0,0,0,0,0,0,0,0,0,0
245,0,0,0,0,0,0,0,0,0,0
210,0,0,0,0,0,0,0,0,0,0
221,1,1,1,1,1,1,1,1,1,1
1,0,0,0,0,0,0,0,0,0,0
116,0,0,0,0,0,0,0,0,0,0
157,1,1,1,1,1,1,1,1,1,1


In [15]:
probs = predictions.mean(axis=1)

In [16]:
accuracy_score(y_test, (probs > .5).astype(int))

0.6933333333333334

## Ensemble evaluation?

Why didn't our score improve?

## Bootstrapping

Let's get started actually making ensemble predictions. However, in order to do that, we'll need to introduce the idea of bootstrapping, or random sampling **with replacement.**

Bootstrapping with `numpy`.

In [17]:
samples = list(range(1,11))
np.random.choice(samples, size=len(samples))

array([6, 6, 7, 7, 2, 8, 8, 6, 5, 6])

Boostrapping with `pandas`:

In [18]:
X_train.sample(n=X_train.shape[0], replace=True, random_state=42)

,Age,Sex,RestBP,Chol,Fbs,RestECG,MaxHR,ExAng,Oldpeak,Slope,Ca,ChestPain_nonanginal,ChestPain_nontypical,ChestPain_typical,Thal_normal,Thal_reversable
59,51,1,125,213,0,2,125,1,1.4,1,1.0,0,0,1,1,0
234,54,0,160,201,0,0,163,0,0.0,1,1.0,1,0,0,1,0
50,41,0,105,198,0,0,168,0,0.0,1,1.0,0,1,0,1,0
117,35,0,138,183,0,0,182,0,1.4,1,0.0,0,0,0,1,0
10,57,1,140,192,0,0,148,0,0.4,2,0.0,0,0,0,0,0
233,74,0,120,269,0,2,121,1,0.2,1,1.0,0,1,0,1,0
291,55,0,132,342,0,0,166,0,1.2,1,0.0,0,1,0,1,0
158,60,1,140,293,0,2,170,0,1.2,2,2.0,0,0,0,0,1
59,51,1,125,213,0,2,125,1,1.4,1,1.0,0,0,1,1,0
136,70,1,145,174,0,0,125,1,2.6,3,0.0,0,0,0,0,1


## Bagging: Bootstrap Aggregating

As we have seen, decision trees are very powerful machine learning models. However, decision trees have some limitations. In particular, trees that are grown very deep tend to learn highly irregular patterns (a.k.a. they overfit their training sets). 

Bagging (bootstrap aggregating) mitigates this problem by exposing different trees to different sub-samples of the training set.

The process for creating bagged decision trees is as follows:
1. From the original data of size $n$, bootstrap $k$ samples each of size $n$ (with replacement!).
2. Build a decision tree on each bootstrapped sample.
3. Make predictions by passing a test observation through all $k$ trees and developing one aggregate prediction for that observation.

![](./assets/Ensemble.png)

### What do you mean by "aggregate prediction?"
As with all of our modeling techniques, we want to make sure that we can come up with one final prediction for our observation. (Building 1,000 trees and coming up with 1,000 predictions for one observation probably wouldn't be very helpful.)

Suppose we want to predict whether or not a Reddit post is going to go viral, where `1` indicates viral and `0` indicates non-viral. We build 100 decision trees. Given a new Reddit post labeled `X_test`, we pass these features into all 100 decision trees.
- 70 of the trees predict that the post in `X_test` will go viral.
- 30 of the trees predict that the post in `X_test` will not go viral.

**What might you expect `.predict(X_test)` and `.predict_proba(X_test)` to do?**
<details><summary> Answer</summary>

- `.predict(X_test)` should output a 1, predicting that the post will go viral.
- `.predict_proba(X_test)` should output 0.7, indicating the probability of the post going viral is 70%.

</details>

## Bagging Classifier using a `for` loop

In the cell below, we'll create an ensemble of trees like before, except this time we'll train each tree to a **bootstrapped** sample of the training data.

In [21]:
predictions = pd.DataFrame(index=X_test.index)
for i in range(1,11):
    X_sample = X_train.sample(n=X_train.shape[0], replace=True)
    y_sample = y_train[X_sample.index]
    t = DecisionTreeClassifier()
    t.fit(X_sample, y_sample)
    predictions['Tree {}'.format(i)] = t.predict(X_test)
predictions.head()

,Tree 1,Tree 2,Tree 3,Tree 4,Tree 5,Tree 6,Tree 7,Tree 8,Tree 9,Tree 10
113,1,1,1,1,1,1,1,1,1,1
195,0,1,1,1,1,1,1,0,1,1
64,1,1,1,1,1,1,1,0,1,1
27,0,1,0,1,0,1,0,0,1,1
245,0,1,1,0,0,0,1,1,1,0


In [22]:
probs = predictions.mean(axis=1)

In [23]:
accuracy_score(y_test, (probs >= .5).astype(int))

0.8266666666666667

## Bagging Classifier using `sklearn`

In the cell below, create and score instance of `BaggingClassifier` on the test set. You should get a similar score to the one in the previous step.

In [34]:
bag = BaggingClassifier()
bag.fit(X_train, y_train)
bag.score(X_test, y_test)

0.76